# BWGNN sobre Yelp-Chi — frente GPU (Google Colab, T4)

Proyecto **fake-review-detector / CheckGraph**. Este notebook responde a una única
pregunta de la Fase 1 (grafo): **¿una GNN entrenada de verdad supera a lo que ya tenemos
con métodos baratos sin entrenamiento (Louvain, OddBall, FRAUDAR, Leiden), y por un margen
que justifique la inversión?**

Implementa **BWGNN** (*Beta Wavelet Graph Neural Network*), de Tang et al., "Rethinking Graph
Neural Networks for Anomaly Detection", ICML 2022 — la idea central del paper es que los nodos
anómalos desplazan el espectro del grafo hacia frecuencias altas, así que un filtro *paso-bajo*
clásico (GCN) los suaviza y los pierde; BWGNN usa un banco de filtros **paso-banda** construido
con wavelets Beta.

## Qué hay que hacer para ejecutarlo

1. `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **T4 GPU** → Guardar.
2. `Entorno de ejecución` → `Ejecutar todo`.
3. Al final se descarga solo un `bwgnn_yelpchi_results.json` con todas las métricas.

**No hay que subir ningún fichero ni instalar nada**: el dataset se descarga solo y el
notebook usa exclusivamente librerías que Colab ya trae (`torch`, `numpy`, `scipy`,
`scikit-learn`).

## Decisión de diseño importante: cero dependencias nuevas

El código oficial de BWGNN usa **DGL**. DGL lleva sin release desde la 2.4.0 y soporta como
mucho PyTorch 2.4, mientras que Colab hoy trae **PyTorch 2.11** — instalarlo aquí es la causa
número uno de que un notebook de este tipo no arranque. PyTorch Geometric 2.8 sí soporta
2.9–2.12, pero tampoco hace falta: BWGNN es, en el fondo, **multiplicación esparsa por
potencias del Laplaciano normalizado**, que `torch.sparse` hace de forma nativa.

Así que aquí hay una **implementación propia en PyTorch puro**, siguiendo la formulación del
paper. Los coeficientes del banco de wavelets Beta se han verificado offline contra la función
`calculate_theta2` del repo oficial (`squareRoot3/Rethinking-Anomaly-Detection`): coinciden
exactamente para todos los grados d = 1..6.

## Cuánto tarda (estimación en T4 del tier gratuito)

| Paso | Tiempo estimado |
|---|---|
| Descarga de `YelpChi.zip` (17 MB) | 10–30 s |
| `scipy.io.loadmat` del `.mat` (198 MB) | 30–90 s |
| Construcción de los 4 Laplacianos esparsos | 10–30 s |
| 8 entrenamientos (4 modelos × 2 regímenes de etiquetas), 200 épocas cada uno | 5–15 min |
| **Total** | **~7–20 min** |

Muy por debajo del límite de sesión de Colab gratuito (~12 h, o ~90 min de inactividad).
La celda opcional del final (barrido de hiperparámetros) añade otros ~10–20 min.

De dónde sale la estimación: la rejilla completa se cronometró en CPU (3 épocas × 8 configs =
184 s), lo que extrapolado a 200 épocas son **~3,4 h en CPU**. El rango de arriba asume que una
T4 acelera la multiplicación esparsa entre 15x y 40x, que es lo típico para un grafo de este
tamaño (~7,7 M aristas × 64 canales) — **no está cronometrado en T4 de verdad**, así que tómalo
como orden de magnitud. Si ves que va mucho más lento, comprueba primero que la celda 1 dijo
que había GPU.

## 1. Comprobación del entorno

Si esta celda dice que **no hay GPU**, para y cambia el entorno de ejecución antes de seguir:
`Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `T4 GPU`. En CPU el notebook
funciona igualmente pero pasa de ~10-15 minutos a unas 3-4 horas.

In [ ]:
import sys, platform

import torch
import numpy as np
import scipy
import sklearn

print("python     ", platform.python_version())
print("torch      ", torch.__version__)
print("numpy      ", np.__version__)
print("scipy      ", scipy.__version__)
print("scikit-learn", sklearn.__version__)
print()

HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    DEVICE = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU DETECTADA: {props.name} ({props.total_memory / 1024**3:.1f} GB)")
    print("Todo listo, puedes ejecutar el resto del notebook.")
else:
    DEVICE = torch.device("cpu")
    print("=" * 72)
    print("  AVISO: NO HAY GPU. Estas en CPU.")
    print()
    print("  Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> T4 GPU")
    print("  y despues Entorno de ejecucion -> Ejecutar todo.")
    print()
    print("  El notebook funciona igual en CPU, pero tardara unas 3-4 horas")
    print("  (cronometrado) en vez de ~10-15 minutos.")
    print("=" * 72)

print("\ndevice =", DEVICE)

## 2. Descarga del dataset Yelp-Chi

`YelpChi.mat` es el grafo preprocesado por los autores de **CARE-GNN** (Dou et al., CIKM 2020)
a partir del dataset original de Rayana & Akoglu. Es exactamente el mismo fichero que
`data.py::load_yelpchi_graph_dataset()` usa en el repo del proyecto, así que las cifras de
aquí son directamente comparables con las que ya tenemos medidas en local.

Contenido del `.mat` (MATLAB 5.0 clásico, `scipy.io.loadmat` lo lee sin `h5py`):

- `net_rur`, `net_rtr`, `net_rsr`: tres grafos homogéneos review–review de 45.954 × 45.954.
  R-U-R = mismo usuario; R-T-R = mismo negocio, mismo rating, misma ventana temporal;
  R-S-R = mismo negocio y mismo rating.
- `homo`: la unión booleana de las tres (verificado en el proyecto).
- `features`: 45.954 × 32, ya normalizadas en [0, 1]. **No son features de este proyecto**,
  son las que calcularon los autores de CARE-GNN.
- `label`: 0 = genuina, 1 = fraude. 6.677 positivos de 45.954 → **base rate 14,53 %**.

In [ ]:
import hashlib
import time
import urllib.request
import zipfile
from pathlib import Path

YELPCHI_URL = "https://github.com/YingtongDou/CARE-GNN/raw/master/data/YelpChi.zip"
DATA_DIR = Path("/content/data") if Path("/content").exists() else Path("./data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DATA_DIR / "YelpChi.zip"
MAT_PATH = DATA_DIR / "YelpChi.mat"

if not MAT_PATH.exists():
    if not ZIP_PATH.exists():
        print("Descargando", YELPCHI_URL)
        t0 = time.time()
        urllib.request.urlretrieve(YELPCHI_URL, ZIP_PATH)
        print(f"  {ZIP_PATH.stat().st_size / 1024**2:.1f} MB en {time.time() - t0:.1f}s")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        names = zf.namelist()
        print("Contenido del zip:", names)
        assert "YelpChi.mat" in names, f"No se encuentra YelpChi.mat en el zip: {names}"
        zf.extract("YelpChi.mat", DATA_DIR)

print(f"\n{MAT_PATH} -> {MAT_PATH.stat().st_size / 1024**2:.1f} MB")

In [ ]:
import scipy.io
import scipy.sparse as sp

t0 = time.time()
mat = scipy.io.loadmat(MAT_PATH)
print(f"loadmat: {time.time() - t0:.1f}s")

features = np.asarray(mat["features"].todense(), dtype=np.float32)
labels = np.asarray(mat["label"]).reshape(-1).astype(np.int64)
RELATIONS = {
    "homo": mat["homo"],
    "net_rur": mat["net_rur"],
    "net_rtr": mat["net_rtr"],
    "net_rsr": mat["net_rsr"],
}

N_NODES, N_FEATS = features.shape
BASE_RATE = float(labels.mean())

# Sanity-checks: si algo de esto falla, el fichero descargado NO es el esperado
# y no tiene sentido seguir (mejor romper aqui que publicar metricas de otra cosa).
assert features.shape == (45954, 32), features.shape
assert labels.shape == (45954,), labels.shape
assert set(np.unique(labels).tolist()) == {0, 1}
assert int(labels.sum()) == 6677, int(labels.sum())
assert 0.0 <= features.min() and features.max() <= 1.0 + 1e-6

print(f"\nnodos           {N_NODES:,}")
print(f"features        {N_FEATS}  (min {features.min():.3f}, max {features.max():.3f})")
print(f"positivos       {int(labels.sum()):,}  ({BASE_RATE:.2%} base rate)")
for k, v in RELATIONS.items():
    print(f"  {k:<9} shape {v.shape}  nnz {v.nnz:,}")

## 3. Laplaciano normalizado como tensor esparso de PyTorch

BWGNN trabaja sobre el Laplaciano normalizado

$$L = I - D^{-1/2} A D^{-1/2}$$

Sobre cada relación se simetriza la adyacencia, se binariza y se añaden self-loops (evita
grados cero en nodos aislados, que en `net_rur` son muchos). El resultado es un
`torch.sparse_coo_tensor`: no hace falta DGL ni PyG para nada, `torch.sparse.mm` propaga igual
y soporta autograd respecto al lado denso.

In [ ]:
def build_laplacian(adj, device):
    # Devuelve L = I - D^-1/2 A D^-1/2 como torch.sparse_coo_tensor.
    # A se simetriza, se binariza y se le anaden self-loops antes de normalizar.
    n = adj.shape[0]
    a = adj.tocoo()
    diag = np.arange(n, dtype=np.int64)
    rows = np.concatenate([a.row.astype(np.int64), a.col.astype(np.int64), diag])
    cols = np.concatenate([a.col.astype(np.int64), a.row.astype(np.int64), diag])
    # tocsr() suma duplicados; despues se binariza a 1.0
    a = sp.coo_matrix((np.ones(rows.size, dtype=np.float32), (rows, cols)), shape=(n, n)).tocsr()
    a.data[:] = 1.0
    a = a.tocoo()

    deg = np.asarray(a.sum(axis=1)).reshape(-1)
    d_inv_sqrt = 1.0 / np.sqrt(np.maximum(deg, 1.0))
    off = -(d_inv_sqrt[a.row] * d_inv_sqrt[a.col]).astype(np.float32)   # -D^-1/2 A D^-1/2

    idx = np.vstack([
        np.concatenate([a.row.astype(np.int64), diag]),
        np.concatenate([a.col.astype(np.int64), diag]),
    ])
    val = np.concatenate([off, np.ones(n, dtype=np.float32)])           # + I
    lap = torch.sparse_coo_tensor(torch.from_numpy(idx), torch.from_numpy(val), (n, n))
    return lap.coalesce().to(device)   # coalesce suma los duplicados de la diagonal


LAPLACIANS = {}
for key, adj in RELATIONS.items():
    t0 = time.time()
    LAPLACIANS[key] = build_laplacian(adj, DEVICE)
    print(f"{key:<9} nnz {LAPLACIANS[key]._nnz():>10,}  ({time.time() - t0:.1f}s)")

# Comprobacion numerica: el Laplaciano normalizado tiene todos sus autovalores en [0, 2],
# asi que la norma de L @ x no puede superar 2 * ||x||. Si esto falla, la normalizacion
# esta mal y todo lo que viene despues no significa nada.
_x = torch.randn(N_NODES, 8, device=DEVICE)
for key, lap in LAPLACIANS.items():
    ratio = (torch.sparse.mm(lap, _x).norm() / _x.norm()).item()
    assert ratio <= 2.0 + 1e-3, (key, ratio)
    print(f"  {key:<9} ||L x|| / ||x|| = {ratio:.3f}  (debe ser <= 2)")
del _x

## 4. Banco de wavelets Beta y el modelo BWGNN

El filtro Beta de orden $(p, q)$ con $p + q = d$ es

$$W_{p,q} = \frac{1}{2^{d}\,B(p+1,\,q+1)} \; L^{p} \, (2I - L)^{q}$$

que, expandido, es un polinomio de grado $d$ en $L$:
$\;W_{p,q} = \sum_k \theta_k L^k$ con

$$\theta_k = \frac{(-1)^{k-p}}{2^{k}\,B(p+1,\,q+1)} \binom{d-p}{\,k-p\,}, \qquad p \le k \le d$$

El repo oficial calcula estos coeficientes con `sympy`; aquí se usa la forma cerrada de arriba
(**verificada numéricamente contra la implementación oficial**, coincidencia exacta para
d = 1..6). Para d = 2 salen los tres filtros
`[3, -3, 0.75]`, `[0, 3, -1.5]`, `[0, 0, 0.75]`.

Optimización respecto al código oficial: como los $d+1$ filtros son polinomios en la **misma**
matriz $L$, las potencias $L^k h$ se calculan **una sola vez** y se reutilizan, en vez de
repropagar por cada filtro. Es matemáticamente idéntico y ahorra multiplicaciones esparsas.

Además de BWGNN se entrenan dos controles que cuestan casi nada y hacen la lectura honesta:

- **MLP** — las mismas 32 features, **sin usar el grafo**. Dice cuánto aporta realmente el grafo.
- **GCN** — la misma arquitectura pero con un filtro **paso-bajo** ($I - L/2$, propagación
  clásica) en vez del banco Beta. Contrasta directamente la tesis del paper: si GCN ≈ MLP
  y BWGNN > ambos, el problema de heterofilia es real y las wavelets Beta lo resuelven.

In [ ]:
import scipy.special
import torch.nn as nn
import torch.nn.functional as F


def beta_wavelet_thetas(d):
    # Coeficientes polinomicos de los d+1 filtros wavelet Beta de orden d.
    thetas = []
    for p in range(d + 1):
        norm = scipy.special.beta(p + 1, d + 1 - p)
        coeffs = [0.0] * (d + 1)
        for k in range(p, d + 1):
            coeffs[k] = float(
                scipy.special.comb(d - p, k - p, exact=True) * ((-1) ** (k - p)) / (2 ** k) / norm
            )
        thetas.append(coeffs)
    return thetas


assert np.allclose(beta_wavelet_thetas(2), [[3.0, -3.0, 0.75], [0.0, 3.0, -1.5], [0.0, 0.0, 0.75]])
print("thetas d=2:", beta_wavelet_thetas(2))
print("thetas d=3:", [[round(c, 4) for c in t] for t in beta_wavelet_thetas(3)])


class BetaWaveletBank(nn.Module):
    # Aplica los d+1 filtros Beta reutilizando las potencias L^k h.

    def __init__(self, d=2):
        super().__init__()
        self.d = d
        self.register_buffer("thetas", torch.tensor(beta_wavelet_thetas(d), dtype=torch.float32))

    @property
    def n_out(self):
        return self.d + 1

    def forward(self, lap, h):
        powers = [h]
        for _ in range(self.d):
            powers.append(torch.sparse.mm(lap, powers[-1]))
        out = []
        for i in range(self.d + 1):
            acc = self.thetas[i, 0] * powers[0]
            for k in range(1, self.d + 1):
                acc = acc + self.thetas[i, k] * powers[k]
            out.append(acc)
        return out


class LowPassBank(nn.Module):
    # Control tipo GCN: propagacion paso-bajo (I - L/2)^d, un solo canal de salida.

    def __init__(self, d=2):
        super().__init__()
        self.d = d

    @property
    def n_out(self):
        return 1

    def forward(self, lap, h):
        for _ in range(self.d):
            h = h - 0.5 * torch.sparse.mm(lap, h)
        return [h]


class GraphNet(nn.Module):
    # MLP de entrada -> banco de filtros por relacion -> concat -> clasificador.
    # Es la arquitectura de BWGNN del paper; cambiando `bank_cls` se obtiene el control GCN.
    # Con varias relaciones (n_rel > 1) es la variante heterogenea: se aplica el mismo
    # banco a cada grafo y se concatenan todas las salidas.

    def __init__(self, in_feats, h_feats, n_rel, bank_cls=BetaWaveletBank, d=2, dropout=0.1):
        super().__init__()
        self.banks = nn.ModuleList([bank_cls(d) for _ in range(n_rel)])
        n_cat = h_feats * sum(b.n_out for b in self.banks)
        self.lin1 = nn.Linear(in_feats, h_feats)
        self.lin2 = nn.Linear(h_feats, h_feats)
        self.lin3 = nn.Linear(n_cat, h_feats)
        self.lin4 = nn.Linear(h_feats, 2)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, laps):
        h = self.drop(F.relu(self.lin1(x)))
        h = F.relu(self.lin2(h))
        outs = []
        for bank, lap in zip(self.banks, laps):
            outs.extend(bank(lap, h))
        h = torch.cat(outs, dim=-1)
        h = self.drop(F.relu(self.lin3(h)))
        return self.lin4(h)


class MLPNet(nn.Module):
    # Control sin grafo: mismas features, ninguna propagacion.

    def __init__(self, in_feats, h_feats, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_feats, h_feats), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h_feats, h_feats), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h_feats, 2),
        )

    def forward(self, x, laps):
        return self.net(x)

## 5. Protocolo de evaluación

Este bloque es el que hace que las cifras signifiquen algo. Tres decisiones explícitas:

**0. Los dos regímenes son los mismos que usa el paper**, así que hay un ancla de cordura
directa: si esta implementación se queda muy por debajo de las cifras publicadas, el problema
está en la implementación, no en el dataset. Tabla 2 del paper, sobre YelpChi:

| | 40 % train | 1 % train |
|---|---|---|
| BWGNN (homo) | AUC **84,03** / F1-macro 71,00 | AUC **72,01** / F1-macro 61,15 |
| BWGNN (hetero) | AUC **90,54** / F1-macro 76,96 | AUC **76,95** / F1-macro 67,02 |

(Nuestro split reserva además un 20 %/1 % de validación, así que no es idéntico al suyo y no
hay que esperar coincidencia al decimal — pero sí el mismo orden de magnitud.)

**1. Split estratificado con `random_state=42`, y las etiquetas de test NO se tocan nunca.**
El modelo entrena solo con las etiquetas de `train`, selecciona época con `val`, y se reporta
`test`. El grafo completo sí se usa en la propagación (evaluación transductiva, como en el
paper y en toda la literatura de este dataset) — pero eso es estructura, no etiqueta.

> **Por qué esto importa aquí**: las señales de grafo que el proyecto ya tiene medidas
> (`features_graph.py`) puntúan cada comunidad de Louvain a partir de **las propias etiquetas**
> de sus miembros (*target encoding*, con leave-one-out). Eso infla el AUC y no es replicable
> en producto, donde no hay etiquetas. El número de este notebook no tiene ese problema:
> es lo que de verdad se podría desplegar. **Por eso no es una comparación limpia contra el
> 0,814 de Louvain — el 0,814 juega con ventaja.**

**2. Dos regímenes de etiquetas**, porque el realista no es el del paper:

| Régimen | train | val | test | Para qué |
|---|---|---|---|---|
| `40pct` | 40 % | 20 % | 40 % | Comparable con el paper (usa 40 %) y con la literatura de Yelp-Chi |
| `1pct` | 1 % | 1 % | 98 % | El escenario de producto: un cliente nuevo casi no tiene etiquetas |

**3. Métricas que no engañan con 14,5 % de positivos.** El ROC-AUC es optimista con clases
desbalanceadas, así que la métrica principal es **Average Precision (PR-AUC)**, que hay que
leer contra la base rate (un modelo aleatorio da AP ≈ 0,145). Y como el producto real revisa
una cola priorizada, se reportan **precisión, recall y lift en el top 1 % / 5 % / 10 %**, que
es lo que un cliente nota.

In [ ]:
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42


def topk_metrics(y_true, scores, frac):
    k = max(1, int(round(len(y_true) * frac)))
    top = np.argsort(-scores)[:k]
    hits = int(y_true[top].sum())
    base = float(y_true.mean())
    precision = hits / k
    return {
        "k": k,
        "precision": precision,
        "recall": hits / max(1, int(y_true.sum())),
        "lift": (precision / base) if base > 0 else float("nan"),
    }


def evaluate(y_true, scores):
    out = {
        "n": int(len(y_true)),
        "n_pos": int(y_true.sum()),
        "base_rate": float(y_true.mean()),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
    }
    for frac in (0.01, 0.05, 0.10):
        out[f"top_{int(frac * 100)}pct"] = topk_metrics(y_true, scores, frac)
    return out


def split_masks(y, train_frac, val_frac, seed=SEED):
    idx = np.arange(len(y))
    tr, rest = train_test_split(idx, train_size=train_frac, stratify=y, random_state=seed)
    va, te = train_test_split(
        rest, train_size=val_frac / (1.0 - train_frac), stratify=y[rest], random_state=seed
    )
    return tr, va, te


REGIMES = {
    "40pct": (0.40, 0.20),
    "1pct": (0.01, 0.01),
}

for name, (trf, vaf) in REGIMES.items():
    tr, va, te = split_masks(labels, trf, vaf)
    assert len(set(tr) & set(te)) == 0 and len(set(va) & set(te)) == 0, "fuga entre splits"
    print(f"{name:<6} train {len(tr):>6,} ({labels[tr].mean():.3%} pos)  "
          f"val {len(va):>6,}  test {len(te):>6,} ({labels[te].mean():.3%} pos)")

## 6. Entrenamiento

Full-batch (45.954 nodos × 32 features cabe de sobra en una T4), Adam, y
**cross-entropy ponderada** por el desbalance de la clase positiva *calculado sobre el train*
— tal y como hace el código oficial.

La época que se reporta es la de **mejor Average Precision en validación**, no la última:
sin esto el resultado depende de dónde se corte el entrenamiento, que es una forma silenciosa
de hacer trampa (o de perder puntos por nada).

Y el **F1-macro se calcula con el umbral que lo maximiza en validación**, no con 0,5. Con
cross-entropy ponderada las probabilidades quedan desplazadas y el 0,5 no significa nada, así
que un F1-macro a 0,5 no sería comparable con la cifra del paper. Se guardan los dos en el JSON
(`f1_macro_val_thr` y `f1_macro_at_0_5`) más el umbral elegido, pero **el comparable es el
primero**. Ni el umbral ni la época miran jamás el conjunto de test.

In [ ]:
MODELS = {
    "mlp_sin_grafo":  dict(kind="mlp",   rels=[]),
    "gcn_homo":       dict(kind="graph", rels=["homo"], bank=LowPassBank),
    "bwgnn_homo":     dict(kind="graph", rels=["homo"], bank=BetaWaveletBank),
    "bwgnn_hetero":   dict(kind="graph", rels=["net_rur", "net_rtr", "net_rsr"], bank=BetaWaveletBank),
}

EPOCHS = 200
HIDDEN = 64
WAVELET_ORDER = 2      # d en el paper
LR = 0.01
WEIGHT_DECAY = 0.0

X = torch.from_numpy(features).to(DEVICE)
Y = torch.from_numpy(labels).to(DEVICE)


def train_one(model_name, regime, epochs=EPOCHS, hidden=HIDDEN, d=WAVELET_ORDER,
              lr=LR, weight_decay=WEIGHT_DECAY, verbose=True):
    cfg = MODELS[model_name]
    train_frac, val_frac = REGIMES[regime]
    tr, va, te = split_masks(labels, train_frac, val_frac)

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    laps = [LAPLACIANS[r] for r in cfg["rels"]]
    if cfg["kind"] == "mlp":
        model = MLPNet(N_FEATS, hidden).to(DEVICE)
    else:
        model = GraphNet(N_FEATS, hidden, len(laps), bank_cls=cfg["bank"], d=d).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_pos = int(labels[tr].sum())
    pos_weight = (len(tr) - n_pos) / max(1, n_pos)
    class_w = torch.tensor([1.0, pos_weight], dtype=torch.float32, device=DEVICE)
    tr_t = torch.from_numpy(tr).to(DEVICE)

    best_ap, best_scores, best_epoch = -1.0, None, -1
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        loss = F.cross_entropy(model(X, laps)[tr_t], Y[tr_t], weight=class_w)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            scores = F.softmax(model(X, laps), dim=1)[:, 1].float().cpu().numpy()
        ap_val = average_precision_score(labels[va], scores[va])
        if ap_val > best_ap:
            best_ap, best_scores, best_epoch = ap_val, scores.copy(), ep
        if verbose and (ep == 1 or ep % 50 == 0):
            print(f"    ep {ep:>4}  loss {loss.item():.4f}  val_AP {ap_val:.4f}")

    # Umbral elegido MAXIMIZANDO F1-macro EN VALIDACION y aplicado a test, que es el
    # criterio del codigo oficial. Con cross-entropy ponderada el 0.5 no significa nada,
    # asi que un F1-macro a 0.5 no seria comparable con la cifra del paper.
    best_thr, best_f1_val = 0.5, -1.0
    for thr in np.unique(np.quantile(best_scores[va], np.linspace(0.01, 0.99, 99))):
        f1_val = f1_score(labels[va], (best_scores[va] >= thr).astype(int),
                          average="macro", zero_division=0)
        if f1_val > best_f1_val:
            best_f1_val, best_thr = f1_val, float(thr)

    res = evaluate(labels[te], best_scores[te])
    res.update(
        model=model_name,
        regime=regime,
        best_epoch=best_epoch,
        val_average_precision=float(best_ap),
        train_seconds=round(time.time() - t0, 1),
        threshold_from_val=best_thr,
        f1_macro_val_thr=float(f1_score(labels[te], (best_scores[te] >= best_thr).astype(int),
                                        average="macro", zero_division=0)),
        f1_macro_at_0_5=float(f1_score(labels[te], (best_scores[te] >= 0.5).astype(int),
                                       average="macro", zero_division=0)),
        n_params=int(sum(p.numel() for p in model.parameters())),
        epochs=epochs, hidden=hidden, wavelet_order=d, lr=lr, weight_decay=weight_decay,
        device=str(DEVICE),
    )
    return res

## 7. La rejilla completa (4 modelos × 2 regímenes)

Aquí es donde se va el grueso del tiempo. En T4 debería tardar entre 5 y 15 minutos en total.
Si la traza de `loss` avanza, está trabajando: no lo pares.

In [ ]:
RESULTS = {}
t_all = time.time()

for regime in REGIMES:
    for model_name in MODELS:
        key = f"{model_name}__{regime}"
        print(f"\n=== {key}")
        RESULTS[key] = train_one(model_name, regime)
        r = RESULTS[key]
        print(f"    -> ROC-AUC {r['roc_auc']:.4f} | AP {r['average_precision']:.4f} | "
              f"P@5% {r['top_5pct']['precision']:.3f} | "
              f"mejor epoca {r['best_epoch']} | {r['train_seconds']}s")

TOTAL_SECONDS = round(time.time() - t_all, 1)
print(f"\nTOTAL: {TOTAL_SECONDS}s ({TOTAL_SECONDS / 60:.1f} min) en {DEVICE}")

## 8. Resultados y comparación con lo que el proyecto ya tiene

Referencias con las que comparar (todas ya medidas o citadas en el repo):

| Referencia | AUC | De dónde sale |
|---|---|---|
| Base rate (azar) | 0,500 | AP de azar = 0,145 |
| Regresión logística sobre las 32 features | 0,658 | sanity-check de `features_graph.py`, sin grafo |
| **Louvain sobre `net_rur`** | **0,814** | `features_graph.py` — ⚠️ *target encoding*, ver abajo |
| Louvain sobre `net_rsr` / `net_rtr` / `net_homo` | 0,636 / 0,587 / 0,572 | idem |
| CARE-GNN (publicado) | ~0,745 | citado en `print_reference_comparison()` |
| SpEagle (publicado, Yelp-NYC) | ~0,78 | otro dataset, referencia indicativa |
| **BWGNN homo (paper, 40 %)** | **0,8403** | Tabla 2 de Tang et al., ICML 2022 |
| **BWGNN hetero (paper, 40 %)** | **0,9054** | idem — el objetivo a batir por esta implementación |

⚠️ **El 0,814 de Louvain no es un rival limpio.** Se calcula puntuando cada comunidad con la
tasa de fraude de sus propios miembros (leave-one-out): usa las etiquetas del conjunto que
evalúa. Es una cifra útil como techo diagnóstico, pero **no es desplegable** — en un cliente
real no hay etiquetas con las que puntuar comunidades. Las cifras de BWGNN de este notebook sí
lo son. Si BWGNN se queda cerca del 0,814 ya es un resultado mejor de lo que el número sugiere;
si lo supera, no hay discusión.

In [ ]:
import pandas as pd

rows = []
for key, r in RESULTS.items():
    rows.append({
        "modelo": r["model"],
        "etiquetas": r["regime"],
        "ROC-AUC": round(r["roc_auc"], 4),
        "AP (PR-AUC)": round(r["average_precision"], 4),
        "AP / base": round(r["average_precision"] / r["base_rate"], 2),
        "P@1%": round(r["top_1pct"]["precision"], 3),
        "P@5%": round(r["top_5pct"]["precision"], 3),
        "P@10%": round(r["top_10pct"]["precision"], 3),
        "R@10%": round(r["top_10pct"]["recall"], 3),
        "lift@5%": round(r["top_5pct"]["lift"], 2),
        "F1-macro": round(r["f1_macro_val_thr"], 3),
        "epoca": r["best_epoch"],
        "seg": r["train_seconds"],
    })

table = pd.DataFrame(rows).sort_values(["etiquetas", "AP (PR-AUC)"], ascending=[True, False])
print(f"Base rate del test: {BASE_RATE:.4f}  (AP de un modelo aleatorio)")
print(f"Un modelo aleatorio daria: ROC-AUC 0.500, AP {BASE_RATE:.3f}, P@k {BASE_RATE:.3f}, lift 1.00x\n")
table

In [ ]:
REFERENCIAS = {
    "azar": 0.500,
    "regresion logistica sobre las 32 features (sin grafo)": 0.658,
    "CARE-GNN (publicado)": 0.745,
    "SpEagle (publicado, Yelp-NYC)": 0.78,
    "Louvain net_rur (target encoding, NO desplegable)": 0.814,
    "BWGNN homo (paper, Tabla 2, 40% train)": 0.8403,
    "BWGNN hetero (paper, Tabla 2, 40% train)": 0.9054,
}
# Cifras publicadas por el paper sobre YelpChi, por regimen (AUC / F1-macro).
PAPER_BWGNN = {
    "40pct": {"bwgnn_homo": (0.8403, 0.7100), "bwgnn_hetero": (0.9054, 0.7696)},
    "1pct":  {"bwgnn_homo": (0.7201, 0.6115), "bwgnn_hetero": (0.7695, 0.6702)},
}

best_key = max(RESULTS, key=lambda k: RESULTS[k]["roc_auc"] if RESULTS[k]["regime"] == "40pct" else -1)
best = RESULTS[best_key]

print("Comparacion de ROC-AUC (regimen 40% de etiquetas)")
print("-" * 62)
for name, auc in sorted(REFERENCIAS.items(), key=lambda kv: kv[1]):
    print(f"  {auc:.4f}   {name}")
print(f"  {best['roc_auc']:.4f}   >>> {best['model']} (este notebook, protocolo limpio) <<<")
print("-" * 62)

delta = best["roc_auc"] - REFERENCIAS["Louvain net_rur (target encoding, NO desplegable)"]
print(f"\nDiferencia frente a Louvain net_rur: {delta:+.4f} AUC")
print("(recordatorio: Louvain juega con ventaja, usa las etiquetas para puntuar comunidades)")

mlp = RESULTS["mlp_sin_grafo__40pct"]
gcn = RESULTS["gcn_homo__40pct"]
print(f"\nCuanto aporta el grafo, de verdad (regimen 40%):")
print(f"  MLP sin grafo        AP {mlp['average_precision']:.4f}  AUC {mlp['roc_auc']:.4f}")
print(f"  GCN (paso-bajo)      AP {gcn['average_precision']:.4f}  AUC {gcn['roc_auc']:.4f}")
print(f"  BWGNN homo           AP {RESULTS['bwgnn_homo__40pct']['average_precision']:.4f}  "
      f"AUC {RESULTS['bwgnn_homo__40pct']['roc_auc']:.4f}")
print(f"  BWGNN hetero         AP {RESULTS['bwgnn_hetero__40pct']['average_precision']:.4f}  "
      f"AUC {RESULTS['bwgnn_hetero__40pct']['roc_auc']:.4f}")
print("\nSi GCN ~= MLP pero BWGNN > ambos, la heterofilia diagnosticada en")
print("INVESTIGACION_GRAFOS.md (ronda 1) queda confirmada y el filtro paso-banda es la causa")
print("de la mejora. Si BWGNN ~= MLP, el grafo de Yelp-Chi no aporta casi nada sobre las")
print("features tabulares, que seria un hallazgo negativo igual de valioso.")

print("\n\nCONTROL DE REPRODUCCION frente a la Tabla 2 del paper")
print("-" * 62)
ok_all = True
for regime, per_model in PAPER_BWGNN.items():
    for model_name, (auc_paper, f1_paper) in per_model.items():
        r = RESULTS[f"{model_name}__{regime}"]
        d_auc = r["roc_auc"] - auc_paper
        flag = "OK " if d_auc > -0.05 else "BAJO"
        ok_all &= (d_auc > -0.05)
        print(f"  [{flag}] {model_name:<14} {regime:<6} "
              f"AUC nuestro {r['roc_auc']:.4f} vs paper {auc_paper:.4f}  ({d_auc:+.4f})   "
              f"F1-macro {r['f1_macro_val_thr']:.4f} vs {f1_paper:.4f}")
print("-" * 62)
if ok_all:
    print("La implementacion reproduce el paper dentro de 5 puntos de AUC en todos los casos.")
else:
    print("Alguna variante se queda >5 puntos de AUC por debajo del paper. Antes de sacar")
    print("conclusiones sobre el dataset, sospechar de la implementacion: probar mas epocas,")
    print("otro learning rate, o d distinto en la celda opcional de barrido.")

## 9. Guardar y descargar el JSON de métricas

El fichero que sale de aquí es lo que hay que traer de vuelta al repo.

In [ ]:
import json

payload = {
    "notebook": "colab/bwgnn_yelpchi.ipynb",
    "dataset": "YelpChi.mat (CARE-GNN preprocessing)",
    "dataset_url": YELPCHI_URL,
    "n_nodes": int(N_NODES),
    "n_features": int(N_FEATS),
    "n_positives": int(labels.sum()),
    "base_rate": BASE_RATE,
    "seed": SEED,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch_version": torch.__version__,
    "total_seconds": TOTAL_SECONDS,
    "protocol": (
        "Split estratificado random_state=42. El modelo entrena solo con las etiquetas de "
        "train, selecciona epoca por AP en val y se reporta test. Sin target encoding: "
        "ninguna etiqueta de test entra en el calculo del score, a diferencia de los scores "
        "de comunidad de features_graph.py."
    ),
    "regimes": {k: {"train_frac": v[0], "val_frac": v[1]} for k, v in REGIMES.items()},
    "reference_aucs": REFERENCIAS,
    "paper_bwgnn_yelpchi_table2": PAPER_BWGNN,
    "results": RESULTS,
}

OUT_PATH = Path("bwgnn_yelpchi_results.json")
OUT_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Guardado en {OUT_PATH.resolve()}  ({OUT_PATH.stat().st_size / 1024:.1f} KB)")
print(json.dumps({k: {"roc_auc": round(v["roc_auc"], 4),
                      "average_precision": round(v["average_precision"], 4)}
                  for k, v in RESULTS.items()}, indent=2))

In [ ]:
try:
    from google.colab import files
    files.download(str(OUT_PATH))
    print("Descarga lanzada. Si el navegador la bloquea, el fichero esta en el panel")
    print("de la izquierda (icono de carpeta) como bwgnn_yelpchi_results.json.")
except ImportError:
    print("No estamos en Colab; el JSON esta en", OUT_PATH.resolve())

---

## 10. (Opcional) Barrido de hiperparámetros de BWGNN

**Viene desactivado** (`RUN_SWEEP = False`), para que `Ejecutar todo` termine en el tiempo
anunciado. Si quieres lanzarlo, pon `RUN_SWEEP = True` y reejecuta las dos celdas de abajo:
añade ~10–20 min en T4. Solo merece la pena si la rejilla principal ha dado un resultado
prometedor y se quiere saber si hay margen de mejora barato. **El notebook ya ha guardado
y descargado su JSON antes de esta celda**, así que saltárselo no pierde nada.

Se barren dos cosas: el **orden `d` del banco de wavelets** (cuántos saltos de propagación
y cuántos filtros) y el **ancho `hidden`**.

### Nota sobre por qué no hay aquí un GAGA o un PC-GNN

Estaban sobre la mesa como segundo modelo. Se han dejado fuera a propósito:
ambos necesitan piezas que no se pueden escribir a ciegas sin poder ejecutarlas
(PC-GNN un sampler de vecinos consciente de la etiqueta con `pick`/`choose` sobre una distancia
aprendida; GAGA un transformer sobre secuencias de vecinos agrupadas por etiqueta, con el
cuidado añadido de no filtrar la etiqueta del propio nodo). Un notebook que arranca y da una
comparación limpia BWGNN vs. GCN vs. MLP vale más que dos modelos a medias. Si BWGNN resulta
ganador, GAGA/PC-GNN son el siguiente notebook, no una celda apresurada de este.

In [ ]:
# Pon esto a True y reejecuta ESTA celda (y la siguiente) para lanzar el barrido.
# Por defecto esta desactivado para que "Ejecutar todo" termine en el tiempo anunciado.
RUN_SWEEP = False

SWEEP = []
if not RUN_SWEEP:
    print("Barrido desactivado (RUN_SWEEP = False).")
    print("Para lanzarlo: pon RUN_SWEEP = True arriba y reejecuta esta celda.")
    sweep_table = None
else:
    for d in (2, 3, 5):
        for hidden in (64, 128):
            print(f"\n=== bwgnn_hetero d={d} hidden={hidden}")
            r = train_one("bwgnn_hetero", "40pct", d=d, hidden=hidden, verbose=False)
            print(f"    ROC-AUC {r['roc_auc']:.4f} | AP {r['average_precision']:.4f} | "
                  f"P@5% {r['top_5pct']['precision']:.3f} | {r['train_seconds']}s")
            SWEEP.append(r)

    sweep_table = pd.DataFrame([{
        "d": r["wavelet_order"], "hidden": r["hidden"],
        "ROC-AUC": round(r["roc_auc"], 4),
        "AP": round(r["average_precision"], 4),
        "P@5%": round(r["top_5pct"]["precision"], 3),
        "epoca": r["best_epoch"], "seg": r["train_seconds"],
    } for r in SWEEP]).sort_values("AP", ascending=False)

    payload["sweep_bwgnn_hetero_40pct"] = SWEEP
    OUT_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\nJSON actualizado con el barrido: {OUT_PATH.resolve()}")

sweep_table

In [ ]:
# Solo tiene sentido si el barrido se ha ejecutado: vuelve a descargar el JSON, ya con el.
if SWEEP:
    try:
        from google.colab import files
        files.download(str(OUT_PATH))
    except ImportError:
        print("JSON final en", OUT_PATH.resolve())
else:
    print("Nada que redescargar: el barrido no se ha ejecutado.")